# Lezione 1 — Dati mancanti: capire il problema, poi risolverlo

**Come si usa questo notebook.** Leggi dall'alto verso il basso ed esegui ogni
cella di codice (Shift+Invio). A metà trovi un esercizio: prova prima da solo
nella cella indicata, poi confronta con la soluzione spiegata subito sotto.
Nell'ultima sezione applichi quello che hai imparato al **progetto del corso**,
che crescerà di un pezzo a ogni lezione.

**Cosa saprai fare alla fine:** davanti a *qualsiasi* tabella con dei buchi —
un sondaggio, un log applicativo, dati clinici, letture di sensori — saprai
decidere in modo motivato cosa scartare, cosa riempire e come non nascondere
le tue decisioni.

## Parte 1 — Teoria: perché i dati mancanti sono un problema universale

Quasi ogni dataset reale ha dei buchi, e le cause sono sempre le stesse in
qualunque dominio:

| Dominio | Perché mancano dati |
|---|---|
| Sondaggi | le persone saltano le domande scomode |
| Medicina | i pazienti abbandonano lo studio |
| Sensori / IoT | la rete perde pacchetti, il sensore si guasta |
| Log applicativi | un campo è stato aggiunto dopo, i vecchi record non ce l'hanno |
| E-commerce | l'utente non compila il campo opzionale |

Il punto teorico fondamentale, che vale ovunque, è questo: **il buco non è
neutro, ha una causa, e la causa determina cosa puoi farci**. Il quadro
classico è di Rubin (1976), che distingue tre meccanismi:

1. **MCAR** (*Missing Completely At Random*) — il buco non dipende da nulla:
   la rete ha perso un pacchetto a caso. Le righe complete sono ancora un
   campione rappresentativo: scartare le righe incomplete non distorce.

2. **MAR** (*Missing At Random*) — il buco dipende da qualcosa che **vedi**
   in altre colonne: le stazioni più vecchie falliscono più spesso, e l'età
   della stazione è nella tabella. Puoi correggerlo perché l'informazione
   sul "perché" è osservabile.

3. **MNAR** (*Missing Not At Random*) — il buco dipende **proprio dal valore
   che manca**: il termometro smette di rispondere quando fa troppo caldo.
   Questo è il caso pericoloso: le righe che vedi sono sistematicamente
   diverse da quelle che non vedi, e scartare o imputare **senza pensarci
   introduce un bias** che nessuna tecnica automatica rimuove.

Da una tabella da sola **non puoi dimostrare** quale meccanismo è all'opera:
serve conoscenza del processo che ha generato i dati. Per questo la regola
professionale è: fai un'assunzione, scrivila, e rendi tracciabile ogni
correzione. Questo principio ti servirà identico quando farai machine
learning, analisi statistiche o dashboard aziendali.

## Parte 2 — Teoria: le strategie e i loro effetti

Le strategie di base sono due, con varianti:

**Scartare** (righe o colonne). Corretto quando la riga ha perso la sua
*identità*: senza id, senza soggetto o senza istante temporale non sai più
*di cosa* stai parlando. Questi sono i **campi critici**. Scartare è
pericoloso quando il meccanismo non è MCAR: butti via proprio le righe
"speciali".

**Imputare** (riempire). Corretto per misure *recuperabili*: un valore
numerico che puoi approssimare con una statistica. Le scelte classiche:

- **media** — usa tutte le distanze, ma è trascinata dai valori estremi;
- **mediana** — resistente agli estremi: con dati asimmetrici o sporchi è
  quasi sempre la scelta più sicura;
- costante, valore più frequente, o modelli predittivi (più avanti nel corso).

Due effetti collaterali da conoscere, perché valgono per **qualsiasi**
imputazione univariata:

1. inserire tante copie dello stesso valore centrale crea un **picco
   artificiale** nella distribuzione e **riduce la varianza** — i dati
   sembrano più "sicuri" di quanto siano;
2. dopo l'imputazione non distingui più i valori veri da quelli inventati —
   a meno di creare **prima** una colonna flag (`x_was_missing`). Il flag
   conserva l'informazione, che spesso è essa stessa predittiva.

Vediamo entrambe le cose con gli occhi.

In [ ]:
import pandas as pd
import numpy as np

# Media vs mediana: cosa succede con un valore estremo
valori = pd.Series([18.2, 18.7, 19.1, 18.9, 41.0])  # 41.0 e' un picco anomalo
print("media  :", round(valori.mean(), 2))
print("mediana:", round(valori.median(), 2))

La media (23.2) non descrive *nessuna* delle giornate tipiche; la mediana
(18.9) sì. Un solo valore estremo ha spostato la media di 4 gradi: ecco
perché "resistente agli estremi" non è un dettaglio, è la proprietà che
guida la scelta.

Ora l'effetto dell'imputazione sulla distribuzione:

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
completo = pd.Series(rng.normal(18, 4, 300))          # la realta'
osservato = completo.copy()
osservato[rng.choice(300, 90, replace=False)] = np.nan  # perdiamo il 30%
imputato = osservato.fillna(osservato.median())

fig, assi = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
assi[0].hist(completo, bins=30)
assi[0].set_title(f"Dati reali (var={completo.var():.1f})")
assi[1].hist(imputato, bins=30)
assi[1].set_title(f"Dopo imputazione mediana (var={imputato.var():.1f})")
plt.tight_layout()
plt.show()

Il picco centrale nel grafico di destra non esiste nella realtà: l'abbiamo
creato noi, e la varianza è scesa. L'imputazione **non è una trasformazione
neutra** — è un compromesso accettabile solo se tracciato con i flag e
dichiarato. Questo concetto ti tornerà identico quando valuterai modelli:
dati "troppo puliti" producono metriche ottimistiche.

## Parte 3 — Esercizio guidato

Ora applichi la teoria a un caso realistico: 120 letture orarie di una rete
di sensori ambientali (temperatura e umidità). Il dataset ha buchi veri e
**non ti dico dove**: la diagnosi è parte dell'esercizio, come nella realtà.

Il tuo compito, nei termini della teoria appena vista:

1. **misura** l'assenza per colonna (prima si misura, poi si decide);
2. **decidi i campi critici** — quali colonne danno identità alla riga?
3. **scarta** le righe che hanno perso l'identità;
4. **imputa** le misure recuperabili con la mediana dei sopravvissuti,
   creando i flag **prima** di riempire.

Il primo passo (la diagnosi) te lo mostro io:

In [ ]:
from pathlib import Path

raw = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'environmental_sensor_missing_challenge.csv')
print(raw.shape)
raw.isna().mean().round(3)  # frazione di valori mancanti per colonna

Leggi il risultato come un medico legge le analisi: ogni colonna ha una
percentuale di buchi. Ora tocca a te. Ragiona così: `reading_id`,
`station_id` e `recorded_at` dicono *quale lettura, dove e quando* — senza
uno di questi la riga non è più interpretabile (campi critici).
`temperature_c` e `humidity_pct` sono misure: recuperabili con la mediana.

**Prova tu nella cella qui sotto**, senza guardare la soluzione. Schema:
copia del DataFrame → `dropna` sui campi critici → flag → `fillna` con la
mediana.

In [ ]:
pulito = raw.copy()

# Scrivi qui il tuo tentativo:
# 1) elimina le righe con reading_id, station_id o recorded_at mancanti
# 2) crea temperature_c_was_missing e humidity_pct_was_missing (PRIMA di imputare)
# 3) riempi temperatura e umidita' con la mediana delle righe rimaste

pulito.head()

### Soluzione spiegata

Leggila riga per riga, confrontandola con il tuo tentativo:

- `dropna(subset=...)` scarta solo le righe senza identità — non tutte
  quelle con un buco qualsiasi: scartare troppo è l'errore più comune;
- i flag vengono creati **prima** di `fillna`, altrimenti l'informazione
  "questo valore era assente" sparisce per sempre;
- la mediana si calcola **dopo** i drop, sui sopravvissuti: le righe
  scartate non devono influenzare la statistica;
- il report finale rende le decisioni verificabili da chiunque.

In [ ]:
CAMPI_CRITICI = ['reading_id', 'station_id', 'recorded_at']

soluzione = raw.copy()
righe_prima = len(soluzione)

soluzione = soluzione.dropna(subset=CAMPI_CRITICI).reset_index(drop=True)

for colonna in ['temperature_c', 'humidity_pct']:
    soluzione[f'{colonna}_was_missing'] = soluzione[colonna].isna()
    soluzione[colonna] = soluzione[colonna].fillna(soluzione[colonna].median())

report = {
    'righe_prima': righe_prima,
    'righe_dopo': len(soluzione),
    'righe_scartate': righe_prima - len(soluzione),
    'temperature_imputate': int(soluzione['temperature_c_was_missing'].sum()),
    'umidita_imputate': int(soluzione['humidity_pct_was_missing'].sum()),
}
assert soluzione[CAMPI_CRITICI].notna().all().all()
assert soluzione[['temperature_c', 'humidity_pct']].notna().all().all()
report

## Parte 4 — Il progetto: Memory AI Lab, passo 1

Da questa lezione in poi costruisci **un unico progetto** che cresce fino a
diventare un sistema completo: il **Memory AI Lab**, un sistema che riceve
"memorie" testuali (eventi, preferenze, fatti), le pulisce, le classifica,
le rende ricercabili e — nelle fasi avanzate del corso — le collega a
embedding, grafi e un modello linguistico.

**Il passo di oggi è l'ingestion**: la porta d'ingresso dei dati. Ogni
record di memoria ha questo schema:

| campo | ruolo |
|---|---|
| `memory_id` | identità → **critico** |
| `text` | il contenuto della memoria → **critico** |
| `timestamp` | quando è successo → **critico** |
| `type` | episodic / preference / semantic → recuperabile (`unknown`) |
| `importance` | punteggio 0–1 → recuperabile (mediana) |

Nella realtà questi record arrivano da pipeline di estrazione che a volte
falliscono a metà: un timeout perde il testo, un parser non trova la data.
È **la stessa teoria di oggi applicata al tuo progetto** — cambia solo la
tabella:

In [ ]:
memorie = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'memory_events_raw.csv')
print('Batch in ingresso:')
memorie

In [ ]:
CRITICI_MEMORIA = ['memory_id', 'text', 'timestamp']

progetto = memorie.dropna(subset=CRITICI_MEMORIA).reset_index(drop=True)

progetto['type_was_missing'] = progetto['type'].isna()
progetto['type'] = progetto['type'].fillna('unknown')

progetto['importance_was_missing'] = progetto['importance'].isna()
progetto['importance'] = progetto['importance'].fillna(progetto['importance'].median())

destinazione = Path('..') / 'datasets' / 'processed' / 'memory_events_clean.csv'
destinazione.parent.mkdir(parents=True, exist_ok=True)
progetto.to_csv(destinazione, index=False)

print(f'Memorie in ingresso : {len(memorie)}')
print(f'Memorie conservate  : {len(progetto)}')
print(f'Scartate (identita) : {len(memorie) - len(progetto)}')
print(f"Type imputati       : {int(progetto['type_was_missing'].sum())}")
print(f"Importance imputate : {int(progetto['importance_was_missing'].sum())}")
progetto

Il progetto ora ha il suo primo componente: un archivio di memorie pulito e
tracciabile, salvato in `datasets/processed/memory_events_clean.csv`. La
prossima lezione partirà **da questo file** e aggiungerà il secondo
componente.

## Cosa hai imparato

- I dati mancanti hanno una **causa** (MCAR/MAR/MNAR) e la causa determina
  la strategia — questo vale per qualsiasi dataset, non solo per questo.
- I **campi critici** danno identità alla riga: se mancano si scarta; le
  misure sono recuperabili: si imputano.
- La **mediana** resiste agli estremi; la media no.
- L'imputazione **altera la distribuzione** (picco artificiale, varianza
  ridotta): va tracciata con flag creati prima di riempire.
- Ogni pulizia produce un **report**: le decisioni nascoste sono bug.

## Quiz (rispondi a mente, poi apri le risposte)

1. Una stazione meteo perde letture soprattutto durante i picchi di calore.
   Che meccanismo è, e perché scartare quelle righe distorce l'analisi?
2. Perché la mediana per l'imputazione si calcola dopo aver scartato le
   righe senza campi critici?
3. Cosa perdi per sempre se crei il flag `was_missing` dopo `fillna`?

<details>
<summary><b>Apri le risposte</b></summary>

1. È MNAR: l'assenza dipende dal valore stesso (temperature alte). Le righe
   rimaste sottorappresentano proprio il caldo estremo, quindi medie e
   modelli costruiti su di esse sottostimano sistematicamente il fenomeno.
2. Perché le righe scartate non fanno più parte del dataset che userai: se
   la loro temperatura entrasse nella mediana, la statistica descriverebbe
   un insieme che non esiste più.
3. L'informazione su *quali* valori erano assenti: dopo `fillna` i valori
   imputati sono indistinguibili dai veri, e non puoi più fare audit né
   usare l'assenza come segnale.
</details>

## Prossima lezione

Arriva un **secondo batch** di memorie per il progetto — e contiene righe
duplicate, numeri scritti come testo e punteggi impossibili. Servono tre
nuove difese: le costruiamo nella Lezione 2.